# 08 — Sélection de features (pipeline gated S4, protocole v3)

« Calculer large, sélectionner étroit » : 35 candidates → 5–8 pour le modèle final.
Étapes (chacune validée avant la suivante) : **② dérédondance** (ici) → ③ label → ③bis probas
régime → ④ sélection importance XGBoost → ⑤ test central VIX vs VIX+spectral.

**② Dérédondance — aucun label, aucun modèle.** Matrice de corrélation feature-feature
(Spearman, robuste aux queues épaisses), clusters de |ρ| > 0,8, on garde la plus interprétable
par cluster. *Look-ahead : nul par construction* (feature-feature ne touche pas la cible) ; on
vérifie en plus la **stabilité** de la structure sur les deux moitiés de l'échantillon.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
df = pd.read_parquet(ROOT / "data" / "processed" / "ml_dataset.parquet")
feats = [c for c in df.columns if c.startswith("f_")]
print(f"{len(feats)} features candidates | {df.shape[0]} jours")
X = df[feats]
cov = X.notna().mean().sort_values()
print("\ncouverture (5 plus faibles) :")
print((cov.head() * 100).round(1).to_string())

35 features candidates | 7221 jours

couverture (5 plus faibles) :
f_sig_rmt_pct    95.3
f_dlam1_63       96.8
f_dlam1_21       97.6
f_dd252          98.6
f_rot21          98.7


In [2]:
# Corrélation Spearman pairwise (complete-observations par paire)
C = X.corr(method="spearman")

# Contrôle de stabilité : |rho| des paires fortes sur 1re vs 2e moitié
half = len(df) // 2
C1 = X.iloc[:half].corr(method="spearman").abs()
C2 = X.iloc[half:].corr(method="spearman").abs()
iu = np.triu_indices(len(feats), 1)
strong = np.abs(C.values[iu]) > 0.8
pairs = [(feats[i], feats[j], C.values[i, j], C1.values[i, j], C2.values[i, j])
         for i, j, s in zip(iu[0], iu[1], strong) if s]
sp = pd.DataFrame(pairs, columns=["f_a", "f_b", "rho_full", "|rho|_1H", "|rho|_2H"])
sp = sp.reindex(sp["rho_full"].abs().sort_values(ascending=False).index)
print(f"paires |rho| > 0,8 : {len(sp)}")
print("stabilité : min(|rho|) sur les deux moitiés parmi ces paires =",
      round(sp[["|rho|_1H", "|rho|_2H"]].min().min(), 2), "(→ structure stable si élevé)")
sp.round(3).to_string(index=False)

paires |rho| > 0,8 : 20
stabilité : min(|rho|) sur les deux moitiés parmi ces paires = 0.68 (→ structure stable si élevé)


'      f_a            f_b  rho_full  |rho|_1H  |rho|_2H\n    f_abs      f_entropy    -0.984     0.983     0.964\n   f_lam1 f_rho_clean252     0.981     0.984     0.965\n f_iv_spx          f_vix     0.978     0.983     0.963\nf_entropy f_rho_clean252    -0.966     0.970     0.976\n   f_lam1      f_entropy    -0.964     0.977     0.955\nf_sig_rmt  f_sig_rmt_pct     0.949     0.900     0.995\n    f_abs f_rho_clean252     0.925     0.940     0.919\n   f_lam1         f_prv1     0.913     0.869     0.944\n   f_lam1          f_abs     0.910     0.939     0.862\n    f_vix      f_iv_comp     0.898     0.876     0.840\n f_iv_spx      f_iv_comp     0.886     0.871     0.779\n   f_prv1 f_rho_clean252     0.879     0.831     0.893\nf_iv_comp     f_iv_xdisp     0.871     0.870     0.838\n f_iv_spx         f_rv63     0.854     0.874     0.773\nf_entropy         f_prv1    -0.853     0.803     0.888\n   f_rv21         f_rv63     0.847     0.867     0.788\n    f_vix         f_rv21     0.836     0.867   

In [3]:
# Clustering hiérarchique : distance = 1 - |rho|, average linkage, coupe à |rho| = 0,8
D = 1.0 - C.abs().values
np.fill_diagonal(D, 0.0)
D = np.nan_to_num(D, nan=1.0)                    # paires sans overlap -> non corrélées
D = (D + D.T) / 2
Z = linkage(squareform(D, checks=False), method="average")
labels = fcluster(Z, t=0.2, criterion="distance")   # 1 - 0.8
clusters = {}
for f, l in zip(feats, labels):
    clusters.setdefault(l, []).append(f)
multi = {k: v for k, v in clusters.items() if len(v) > 1}
singles = [v[0] for v in clusters.values() if len(v) == 1]
print(f"{len(clusters)} clusters : {len(multi)} redondants, {len(singles)} features isolées\n")
for k, v in sorted(multi.items(), key=lambda kv: -len(kv[1])):
    print(f"cluster redondant ({len(v)}) : {v}")
print(f"\nfeatures isolées (gardées d'office) : {sorted(singles)}")

26 clusters : 3 redondants, 23 features isolées

cluster redondant (5) : ['f_lam1', 'f_abs', 'f_entropy', 'f_prv1', 'f_rho_clean252']
cluster redondant (5) : ['f_iv_spx', 'f_vix', 'f_rv21', 'f_rv63', 'f_iv_comp']
cluster redondant (2) : ['f_sig_rmt', 'f_sig_rmt_pct']

features isolées (gardées d'office) : ['f_anchor_gap', 'f_cost_era', 'f_dd252', 'f_div_21', 'f_dlam1_21', 'f_dlam1_63', 'f_drho_21', 'f_drho_imp_21', 'f_iv_xdisp', 'f_k', 'f_lam2', 'f_mom63', 'f_rho_imp', 'f_rho_trail63', 'f_rot21', 'f_sig_base', 'f_skew50', 'f_term_slope', 'f_turb', 'f_turb21', 'f_volofvol', 'f_vrp', 'f_xdisp']


In [4]:
# DÉCISION ② (règle pré-fixée : la plus interprétable/standard, niveau économique direct > dérivé)
import json

KEEP = {
    "bloc force du mode marché": dict(
        survivant="f_lam1",
        eliminees={"f_abs": "corr 0,91 avec λ₁/N (top-5 ≈ λ₁ + bruit)",
                   "f_entropy": "corr −0,96 : l'entropie EST (1−λ₁/N) reparamétrée",
                   "f_prv1": "corr 0,91 : participation ≈ largeur du même mode",
                   "f_rho_clean252": "corr 0,98 : l'ancre lente ≈ part du mode marché — info conservée via f_anchor_gap (isolée)"},
        justification="λ₁/N = LA quantité RMT canonique (Laloux/BBP), narrative thèse directe, spec HMM utilisateur",
    ),
    "bloc niveau de vol": dict(
        survivant="f_vix",
        eliminees={"f_iv_spx": "corr 0,978 avec VIX (91j ATM vs 30j MFIV : même facteur niveau)",
                   "f_iv_comp": "corr 0,90/0,89 avec VIX/IV-SPX",
                   "f_rv21": "corr 0,84 : la RV suit le niveau ; l'écart utile est déjà f_vrp (isolée)",
                   "f_rv63": "corr 0,83-0,85 : idem"},
        justification="le VIX officiel = requis par le TEST CENTRAL « VIX seul » (crédibilité littérature) ; f_vrp garde l'écart IV−RV",
    ),
    "bloc signal rmt": dict(
        survivant="f_sig_rmt",
        eliminees={"f_sig_rmt_pct": "corr 0,95 : rang expanding = transformation monotone du signal une fois l'historique long"},
        justification="niveau économique direct (l'input de la porte v1_rmt) > dérivé normalisé",
    ),
}

eliminated = [f for v in KEEP.values() for f in v["eliminees"]]
survivors_clusters = [v["survivant"] for v in KEEP.values()]
candidates = sorted(set(feats) - set(eliminated))
assert set(survivors_clusters) <= set(candidates) and len(candidates) == len(feats) - len(eliminated)
print(f"35 candidates -> {len(eliminated)} éliminées -> {len(candidates)} retenues pour l'étape ④ :\n")
for k, v in KEEP.items():
    print(f"[{k}] survivant : {v['survivant']}  — {v['justification']}")
    for f, why in v["eliminees"].items():
        print(f"   ✗ {f} : {why}")
print(f"\nliste post-dérédondance ({len(candidates)}) : {candidates}")

with open(ROOT / "data" / "processed" / "ml_candidates_postdedup.json", "w") as fh:
    json.dump({"candidates": candidates, "eliminated": eliminated,
               "rule": "clusters |rho|>0.8, survivant le plus interprétable (notebook 08, étape 2)"}, fh, indent=2)
print("\n→ data/processed/ml_candidates_postdedup.json écrit")

35 candidates -> 9 éliminées -> 26 retenues pour l'étape ④ :

[bloc force du mode marché] survivant : f_lam1  — λ₁/N = LA quantité RMT canonique (Laloux/BBP), narrative thèse directe, spec HMM utilisateur
   ✗ f_abs : corr 0,91 avec λ₁/N (top-5 ≈ λ₁ + bruit)
   ✗ f_entropy : corr −0,96 : l'entropie EST (1−λ₁/N) reparamétrée
   ✗ f_prv1 : corr 0,91 : participation ≈ largeur du même mode
   ✗ f_rho_clean252 : corr 0,98 : l'ancre lente ≈ part du mode marché — info conservée via f_anchor_gap (isolée)
[bloc niveau de vol] survivant : f_vix  — le VIX officiel = requis par le TEST CENTRAL « VIX seul » (crédibilité littérature) ; f_vrp garde l'écart IV−RV
   ✗ f_iv_spx : corr 0,978 avec VIX (91j ATM vs 30j MFIV : même facteur niveau)
   ✗ f_iv_comp : corr 0,90/0,89 avec VIX/IV-SPX
   ✗ f_rv21 : corr 0,84 : la RV suit le niveau ; l'écart utile est déjà f_vrp (isolée)
   ✗ f_rv63 : corr 0,83-0,85 : idem
[bloc signal rmt] survivant : f_sig_rmt  — niveau économique direct (l'input de la porte v1_r

In [5]:
# RESSERRAGE À 10 (jugement économique, autorisé par l'utilisateur — plus robuste que
# l'importance in-sample sur 12 crises, qui sur-apprendrait le choix des features).
# HMM/GMM regime prob AJOUTÉE PAR-DESSUS quoi qu'il arrive (11e input) — directive utilisateur.
# f_cost_era SORTIE des prédicteurs -> réservée à la PORTE coût-aware (caveat calibration full-sample).
FINAL10 = {
    # --- baseline vol (bras « VIX seul » du test central) ---
    "f_vix":          "niveau de vol officiel — baseline littérature du test central",
    "f_term_slope":   "structure de terme 30->91j — l'indicateur de stress le plus documenté",
    # --- spectral RMT (la contribution TESTÉE au-delà du VIX) ---
    "f_lam1":         "λ₁/N, force du mode marché — quantité RMT canonique",
    "f_dlam1_63":     "dynamique de λ₁/N sur 63j — la corrélation qui MONTE (précurseur, horizon-cohérent)",
    "f_turb21":       "turbulence de Mahalanobis sur l'inverse NETTOYÉE — là où le clipping RMT a de la valeur",
    "f_k":            "nb de facteurs > bord Laloux — largeur du régime (distinct du niveau λ₁)",
    "f_rot21":        "rotation du vecteur propre dominant — CHANGEMENT de régime (orthogonal au niveau)",
    # --- économique / signal ---
    "f_vrp":          "variance risk premium IV−RV — prédicteur de crash documenté",
    "f_drho_imp_21":  "variation de la corr IMPLICITE — le marché COMMENCE à pricer le stress",
    "f_sig_rmt":      "le signal primaire lui-même — contexte de méta-labeling (force du pari proposé)",
}
final10 = list(FINAL10)
assert len(final10) == 10 and set(final10) <= set(candidates)
core_hmm = ["f_lam1", "f_dlam1_63", "f_turb21", "f_vix"]   # 3-4 features pour HMM/GMM (spec v3)
assert set(core_hmm) <= set(final10)

print("10 features finales (jugement, label-free) :")
for f, why in FINAL10.items():
    print(f"  {f:16s} {why}")
print(f"\n+ proba régime HMM/GMM (stacking, par-dessus) = 11 inputs")
print(f"core set HMM/GMM (3-4) : {core_hmm}")
print(f"f_cost_era -> porte coût-aware uniquement (hors prédiction)")

with open(ROOT / "data" / "processed" / "ml_features_final.json", "w") as fh:
    json.dump({"final10": final10, "core_hmm": core_hmm,
               "cost_gate_only": "f_cost_era",
               "rule": "jugement économique + diversité blocs (notebook 08, étape 2->10)"}, fh, indent=2)
print("\n→ data/processed/ml_features_final.json écrit")

10 features finales (jugement, label-free) :
  f_vix            niveau de vol officiel — baseline littérature du test central
  f_term_slope     structure de terme 30->91j — l'indicateur de stress le plus documenté
  f_lam1           λ₁/N, force du mode marché — quantité RMT canonique
  f_dlam1_63       dynamique de λ₁/N sur 63j — la corrélation qui MONTE (précurseur, horizon-cohérent)
  f_turb21         turbulence de Mahalanobis sur l'inverse NETTOYÉE — là où le clipping RMT a de la valeur
  f_k              nb de facteurs > bord Laloux — largeur du régime (distinct du niveau λ₁)
  f_rot21          rotation du vecteur propre dominant — CHANGEMENT de régime (orthogonal au niveau)
  f_vrp            variance risk premium IV−RV — prédicteur de crash documenté
  f_drho_imp_21    variation de la corr IMPLICITE — le marché COMMENCE à pricer le stress
  f_sig_rmt        le signal primaire lui-même — contexte de méta-labeling (force du pari proposé)

+ proba régime HMM/GMM (stacking, par-dess